# Entraînement YOLOv8 — Conduite de Nuit
**Projet IA — Analyseur Intelligent de Scènes de Conduite**

Ce notebook entraîne et compare deux modèles YOLOv8 sur le dataset BDD100K filtré **Conduite de Nuit** :
- **YOLOv8n** — léger, rapide
- **YOLOv8s** — plus précis

Particularités nocturnes :
- Augmentations sur la luminosité (`hsv_v=0.6`) pour simuler différents éclairages
- Seuil de confiance réduit à 0.30 (piétons peu visibles la nuit)
- Mosaic + mixup pour la robustesse

In [ ]:
# - Installation des dépendances  -
# !pip install ultralytics pandas matplotlib seaborn -q

In [ ]:
# ============================================================
# 0. Imports
# ============================================================
import os
import glob
import shutil
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO, checks
from IPython.display import display

checks()  # Vérifie GPU, CUDA, ultralytics
print('✅ Imports OK')

In [ ]:
# ============================================================
# 1. Configuration des chemins
# ============================================================

#  Adapter si on utilise Google Colab :
# DATASET_DIR = '/content/drive/MyDrive/conduite_nuit'
# DATA_YAML   = '/content/drive/MyDrive/data.yaml'

#là on travaille sur visual studio code
DATASET_DIR = '../data/conduite_nuit'
DATA_YAML   = '../data/data.yaml'

train_imgs = glob.glob(f'{DATASET_DIR}/images/train/*.jpg')
val_imgs   = glob.glob(f'{DATASET_DIR}/images/val/*.jpg')

print(f'📁 Dataset : {DATASET_DIR}')
print(f'   Images train : {len(train_imgs)}')
print(f'   Images val   : {len(val_imgs)}')
print(f'   data.yaml    : {"✅ trouvé" if os.path.exists(DATA_YAML) else "❌ manquant — exécuter convert_dataset.py"}')

In [ ]:
# ============================================================
# 2. Split val automatique (si val/ est vide)
# ============================================================
if len(val_imgs) == 0 and len(train_imgs) > 0:
    print('⚠️  Dossier val/ vide — création automatique du split validation (20%)...')
    os.makedirs(f'{DATASET_DIR}/images/val', exist_ok=True)
    os.makedirs(f'{DATASET_DIR}/labels/val',  exist_ok=True)
    
    random.seed(42)
    random.shuffle(train_imgs)
    val_split = train_imgs[:int(len(train_imgs) * 0.2)]
    
    for img_path in val_split:
        fname    = os.path.basename(img_path)
        lbl_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
        shutil.move(img_path, f'{DATASET_DIR}/images/val/{fname}')
        if os.path.exists(lbl_path):
            shutil.move(lbl_path, f'{DATASET_DIR}/labels/val/{fname.replace(".jpg",".txt")}')
    
    # Recompter
    train_imgs = glob.glob(f'{DATASET_DIR}/images/train/*.jpg')
    val_imgs   = glob.glob(f'{DATASET_DIR}/images/val/*.jpg')
    print(f'✅ Split créé : {len(val_imgs)} images val, {len(train_imgs)} images train')
else:
    print('✅ Split train/val déjà existant')

In [ ]:
# ============================================================
# 3. Entraînement — YOLOv8n (léger)
# Augmentations adaptées à la nuit
# ============================================================
print('🚀 Entraînement YOLOv8n...')
model_n = YOLO('yolov8n.pt')  # pretrained COCO

results_n = model_n.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,           # réduire à 8 si OOM
    name     = 'conduite_nuit_yolov8n',
    project  = '../runs/train',
    patience = 10,           # early stopping
    # ─ Augmentations nocturnes ─
    hsv_h    = 0.010,        # teinte : faible variation (peu de couleurs la nuit)
    hsv_s    = 0.4,          # saturation réduite
    hsv_v    = 0.6,          # luminosité : forte variation (simuler différents éclairages)
    fliplr   = 0.5,          # flip horizontal
    mosaic   = 1.0,          # mosaïque 4 images
    mixup    = 0.1,          # mixup léger
    exist_ok = True,
)
print('✅ YOLOv8n entraîné')

In [ ]:
# ============================================================
# 4. Entraînement — YOLOv8s (plus précis)
# ============================================================
print('🚀 Entraînement YOLOv8s...')
model_s = YOLO('yolov8s.pt')

results_s = model_s.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = 'conduite_nuit_yolov8s',
    project  = '../runs/train',
    patience = 10,
    hsv_h    = 0.010,
    hsv_s    = 0.4,
    hsv_v    = 0.6,
    fliplr   = 0.5,
    mosaic   = 1.0,
    mixup    = 0.1,
    exist_ok = True,
)
print('✅ YOLOv8s entraîné')

In [ ]:
# ============================================================
# 5. Évaluation — comparaison des métriques
# ============================================================
def get_metrics(model_path, data_yaml):
    """Évalue un modèle et retourne ses métriques principales."""
    model   = YOLO(model_path)
    metrics = model.val(data=data_yaml, split='val', verbose=False)
    return {
        'mAP@0.5':      round(metrics.box.map50, 4),
        'mAP@0.5:0.95': round(metrics.box.map,   4),
        'Précision':    round(metrics.box.mp,     4),
        'Rappel':       round(metrics.box.mr,     4),
    }

metrics_n = get_metrics('../runs/train/conduite_nuit_yolov8n/weights/best.pt', DATA_YAML)
metrics_s = get_metrics('../runs/train/conduite_nuit_yolov8s/weights/best.pt', DATA_YAML)

df_metrics = pd.DataFrame(
    [metrics_n, metrics_s],
    index=['YOLOv8n', 'YOLOv8s']
)

print('\n📊 Comparaison des métriques — Conduite de Nuit\n')
display(df_metrics.style.highlight_max(axis=0, color='#d4edda').format('{:.4f}'))

In [ ]:
# ============================================================
# 6. Courbes d'entraînement (loss + métriques par epoch)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#f8f9fa')

configs = [
    (axes[0], 'conduite_nuit_yolov8n', 'YOLOv8n', '#2c3e50'),
    (axes[1], 'conduite_nuit_yolov8s', 'YOLOv8s', '#3498db'),
]

for ax, run_name, label, color in configs:
    csv_path = f'../runs/train/{run_name}/results.csv'
    if os.path.exists(csv_path):
        df_res = pd.read_csv(csv_path)
        df_res.columns = df_res.columns.str.strip()
        epochs = df_res['epoch']
        ax.plot(epochs, df_res['metrics/mAP50(B)'],    label='mAP@0.5',      color='#3498db', lw=2)
        ax.plot(epochs, df_res['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='#e74c3c', lw=2)
        ax.plot(epochs, df_res['metrics/precision(B)'],label='Précision',     color='#2ecc71', ls='--')
        ax.plot(epochs, df_res['metrics/recall(B)'],   label='Rappel',        color='#f39c12', ls='--')
        ax.set_title(f'{label} — Conduite de Nuit', fontsize=12)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{label}\nnon encore entraîné',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)

plt.suptitle('Courbes d\'entraînement — YOLOv8n vs YOLOv8s (Conduite de Nuit)', fontsize=13)
plt.tight_layout()
os.makedirs('../data/conduite_nuit', exist_ok=True)
plt.savefig('../data/conduite_nuit/courbes_entrainement.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 7. Bar chart comparaison finale
# ============================================================
metrics_keys = list(df_metrics.columns)
x     = range(len(metrics_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars_n = ax.bar([i - width/2 for i in x], df_metrics.loc['YOLOv8n'],
                width, label='YOLOv8n', color='#2c3e50', alpha=0.85)
bars_s = ax.bar([i + width/2 for i in x], df_metrics.loc['YOLOv8s'],
                width, label='YOLOv8s', color='#3498db', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(metrics_keys, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_title('Comparaison YOLOv8n vs YOLOv8s — Conduite de Nuit', fontsize=13)
ax.set_ylabel('Score')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in [*bars_n, *bars_s]:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
            f'{h:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/conduite_nuit/comparaison_modeles.png', dpi=150)
plt.show()

# Conclusion
best = 'YOLOv8s' if df_metrics.loc['YOLOv8s', 'mAP@0.5'] >= df_metrics.loc['YOLOv8n', 'mAP@0.5'] else 'YOLOv8n'
print(f'\n✅ Meilleur modèle pour Conduite de Nuit : {best}')
print(f'   → runs/train/conduite_nuit_{best.lower()}/weights/best.pt')

In [ ]:
# ============================================================
# 8. Prédictions visuelles sur images de validation nocturnes
# ============================================================
best_model = YOLO('../runs/train/conduite_nuit_yolov8s/weights/best.pt')
val_images = glob.glob(f'{DATASET_DIR}/images/val/*.jpg')[:6]

if val_images:
    results = best_model.predict(
        val_images,
        conf    = 0.30,
        save    = True,
        project = '../runs/predict',
        name    = 'nuit_preview',
        exist_ok= True,
    )

    pred_imgs = sorted(glob.glob('../runs/predict/nuit_preview/*.jpg'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor('#1a1a2e')

    for ax, path in zip(axes.flat, pred_imgs):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(os.path.basename(path), fontsize=7, color='white')
        ax.axis('off')

    # Cacher les axes inutilisés
    for ax in axes.flat[len(pred_imgs):]:
        ax.set_visible(False)

    plt.suptitle('Prédictions YOLOv8s — Conduite de Nuit (conf ≥ 0.30)',
                 fontsize=13, color='white')
    plt.tight_layout()
    plt.savefig('../data/conduite_nuit/predictions_val.png', dpi=150,
                facecolor=fig.get_facecolor())
    plt.show()
else:
    print('⚠️ Aucune image de validation trouvée — vérifiez DATASET_DIR')

## En Résumé

Après l'entraînement, nos modèles sont disponibles dans :
```
runs/train/conduite_nuit_yolov8n/weights/best.pt   ← YOLOv8n
runs/train/conduite_nuit_yolov8s/weights/best.pt   ← YOLOv8s (recommandé plus précis)
```

L'interface Streamlit les chargera automatiquement.